In [1]:
using CSV, DataFrames, Dates, JSON, XLSX, JuMP, Gurobi
include("NORMAL/model_amrp_new.jl")
include("build_graph.jl")

graph_reduction (generic function with 1 method)

In [2]:
function computation_new(df_flights, nbr_ac)    #set_optimizer_attribute(model, "LogToConsole", 0)

    # Agrégation par jour pour obtenir les totaux quotidiens
    daily_stats = combine(groupby(df_flights, :DAY),
        :AIR_TIME => sum => :FLYING_TIME,
        :AIR_TIME => length => :TAKEOFF
    )
    
    # Calcul des moyennes par avion et par jour
    total_flying_time = sum(daily_stats.FLYING_TIME)
    total_takeoffs = sum(daily_stats.TAKEOFF)
    nbr_days = length(unique(df_flights.DAY))
    fh_ac_day = round(Int, total_flying_time / nbr_ac / nbr_days)
    tk_ac_day = round(Int, total_takeoffs / nbr_ac / nbr_days)
    fh_tk = round(Int, total_flying_time / total_takeoffs)
    
    return (fh_ac_day = fh_ac_day, tk_ac_day = tk_ac_day, fh_tk = fh_tk)
end

function build_aircraft_path(start_aircraft::String, succ::Dict{String, String})
    """Construit le chemin complet d'un avion à partir du graphe des successeurs"""
    chemin = ["s", start_aircraft]
    current = start_aircraft    #set_optimizer_attribute(model, "LogToConsole", 0)

    
    while haskey(succ, current) && succ[current] != "t"
        current = succ[current]
        push!(chemin, current)
   end
    # Ajouter le nœud terminal si accessible
    haskey(succ, current) && push!(chemin, succ[current])
    
    return chemin
end 

mutable struct flight
    orig::String
    dest::String
    dt::Int
    at::Int
end 


In [5]:
env = Gurobi.Env()
fold = "INSTANCES/instances_literature_xlsx/A_MTN_3/"
for i in 11:20
    fold_json = "INSTANCES/instances_literature_json/A_MTN_3/"
    inst_name = "354FL_8A_"*string(i)
    inst_path = fold_json*inst_name

    fold_1 = "NORMAL/RESULTS_NOR/A_MTN_1/"
    Outputs_fold = fold_1
    instance_file = inst_path*".json"

    instance_data = build_graph(instance_file)
    path_file = Outputs_fold*"result_"*inst_name*"_RF.txt"

    solution = model_amrp(env, instance_data, path_file, 8, true, false, 20, true, false, false, false, "MIN", "MIN")
    sx = solution.x

    graph = instance_data.graph
    node_sets = graph.node_sets
    arc_sets = graph.arc_sets
    fl_data = instance_data.fl_data
    other_data = instance_data.other_data

    A = graph.arcs
    a_nodes = node_sets.ac_nodes
    aircraft_paths = Dict{String, Vector{String}}()
    succ = Dict{String, String}()
    for (i, j) in A
        if sx[(i, j)] >= 0.9
            succ[i] = j
            #println("  Arc selected: $(i) -> $(j)")
        end
    end

    for aircraft in a_nodes
        #println(succ)
        chemin = build_aircraft_path(aircraft, succ)
        #println(chemin)
        aircraft_paths[aircraft] = chemin
    end

    ac_rm = ["N125", "N126", "N127", "N128"]
    nbr_ac = 4
    fl_to_remove = []
    for ac in ac_rm
        chemin = aircraft_paths[ac][3:end-1]
        for elt in chemin
            part = split(elt, '_')
            push!(fl_to_remove, (part[1], part[2], parse(Int, part[3]), parse(Int, part[4])))
        end
    end 

    df_flights = DataFrame(XLSX.readtable(fold*inst_name*".xlsx", "Data"))

    result = computation_new(df_flights, nbr_ac)    
    df_param = DataFrame(TRT = 35, F = 6000, MT = 480, T = 50, D = 10, NBR_TP = 7, FH_TK = result.fh_tk, FH_DAY = result.fh_ac_day, TK_DAY = result.tk_ac_day)

    filter!(row -> (row.ORIGIN_AIRPORT, row.DESTINATION_AIRPORT, row.DEPARTURE_TIME, row.ARRIVAL_TIME) ∉ fl_to_remove, df_flights)
    df_ac = DataFrame(XLSX.readtable(fold*inst_name*".xlsx", "Aircrafts"))
    df_ac = df_ac[1:4, :]
    
    nbr_fl = nrow(df_flights)
    result = computation_new(df_flights, nbr_ac)
    df_param.FH_DAY = [result.fh_ac_day]
    df_param.FH_TK = [result.fh_tk]
    df_param.TK_DAY = [result.tk_ac_day]

    airport = unique(vcat(df_flights.ORIGIN_AIRPORT, df_flights.DESTINATION_AIRPORT))
    nbr_airport = length(airport) 
    df_mstation = DataFrame(MTN_STATIONS = airport)
    for j in 1:7
        cap_mat = [rand() < 0.2 ? 0 : rand(1:3) for l in 1:nbr_airport]
        df_mstation[!, "T_" * string(j)] = cap_mat
    end
    filename = fold*string(nbr_fl)*"FL_"*string(nbr_ac)*"A_"*string(i)*".xlsx"
    XLSX.openxlsx(filename, mode = "w") do xf
        # Supprimer la feuille par défaut "Sheet1"
        XLSX.rename!(xf["Sheet1"], "Data")
        data_sheet = xf["Data"]

        #data_sheet = XLSX.addsheet!(xf, "Data")
        parameters_sheet = XLSX.addsheet!(xf, "Parameters")
        mtn_stations_sheet = XLSX.addsheet!(xf, "M_stations")
        aircrafts_sheet = XLSX.addsheet!(xf, "Aircrafts")

        XLSX.writetable!(data_sheet, Tables.columntable(df_flights); write_columnnames = true)
        XLSX.writetable!(parameters_sheet, Tables.columntable(df_param); write_columnnames = true)
        XLSX.writetable!(mtn_stations_sheet, Tables.columntable(df_mstation); write_columnnames = true)
        XLSX.writetable!(aircrafts_sheet, Tables.columntable(df_ac); write_columnnames = true)
        println("Fichier xlsx créé")
    end
end 

Set parameter Username
Set parameter LicenseID to value 2765716
Academic license - for non-commercial use only - expires 2027-01-14
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : OPTIMAL
Fichier xlsx créé
Set parameter Threads to value 8
STATUT : TIME_LIMIT
Fichier xlsx créé
